# Postcode Population Model Phase 1

Postinumeroaluekohtainen synteettinen väestö UK-kotitalousmallilla + PAAVO-metatiedot.

In [ ]:
import sys
from pathlib import Path
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

cwd = Path.cwd().resolve()
repo_root = next((p for p in [cwd, *cwd.parents] if (p / "extensions" / "scenario_api").exists()), None)
if repo_root is None:
    raise RuntimeError("Could not locate extensions/scenario_api from current working directory")

extensions_dir = repo_root / "extensions"
if str(extensions_dir) not in sys.path:
    sys.path.insert(0, str(extensions_dir))

from scenario_api import (
    PopulationSynthesisConfig,
    OPENABM_AGE_BIN_LABELS,
    load_paavo_postcode_targets,
    build_postcode_population,
)


## Controls

In [ ]:
# Input paths (replace with real data when available)
paavo_input_path = repo_root / "data" / "processed" / "paavo_postcode_demo.csv"
uk_households_input_path = repo_root / "data" / "processed" / "uk_household_templates_demo.csv"

# Reproducibility
random_seed = 1234

# Sampling config
config = PopulationSynthesisConfig(
    random_seed=random_seed,
    max_iterations_per_postcode=120000,
    phase1_fraction=0.70,
    soft_accept_prob_phase1=0.03,
    soft_accept_prob_phase2=0.002,
    overshoot_fraction_phase1=0.12,
    overshoot_fraction_phase2=0.03,
    population_tolerance=6,
    sse_threshold=0.0,
    enable_household_swap_refinement=True,
    swap_iterations_per_postcode=30000,
)

print("paavo_input_path:", paavo_input_path)
print("uk_households_input_path:", uk_households_input_path)
print("random_seed:", random_seed)
print("swap_refinement_enabled:", config.enable_household_swap_refinement)
print("swap_iterations_per_postcode:", config.swap_iterations_per_postcode)
print("age_bins:", OPENABM_AGE_BIN_LABELS)


## Create Local Demo Inputs (If Missing)

In [ ]:
if not paavo_input_path.exists():
    paavo_input_path.parent.mkdir(parents=True, exist_ok=True)
    paavo_demo = pd.DataFrame([
        {
            "postcode": "00100",
            "postcode_name": "Helsinki Keskusta",
            "municipality": "Helsinki",
            "region": "Uusimaa",
            "latitude": 60.1699,
            "longitude": 24.9384,
            "age_0_6": 180,
            "age_7_12": 150,
            "age_13_17": 120,
            "age_18_24": 310,
            "age_25_34": 520,
            "age_35_44": 430,
            "age_45_54": 350,
            "age_55_64": 290,
            "age_65_74": 230,
            "age_75_84": 170,
            "age_85_plus": 90,
        },
        {
            "postcode": "00200",
            "postcode_name": "Lauttasaari",
            "municipality": "Helsinki",
            "region": "Uusimaa",
            "latitude": 60.1580,
            "longitude": 24.8807,
            "age_0_6": 210,
            "age_7_12": 200,
            "age_13_17": 170,
            "age_18_24": 260,
            "age_25_34": 480,
            "age_35_44": 500,
            "age_45_54": 410,
            "age_55_64": 330,
            "age_65_74": 260,
            "age_75_84": 180,
            "age_85_plus": 95,
        },
        {
            "postcode": "00300",
            "postcode_name": "Meilahti",
            "municipality": "Helsinki",
            "region": "Uusimaa",
            "latitude": 60.1907,
            "longitude": 24.9069,
            "age_0_6": 160,
            "age_7_12": 140,
            "age_13_17": 115,
            "age_18_24": 240,
            "age_25_34": 410,
            "age_35_44": 390,
            "age_45_54": 320,
            "age_55_64": 280,
            "age_65_74": 220,
            "age_75_84": 150,
            "age_85_plus": 80,
        },
        {
            "postcode": "00400",
            "postcode_name": "Munkkiniemi",
            "municipality": "Helsinki",
            "region": "Uusimaa",
            "latitude": 60.2015,
            "longitude": 24.8774,
            "age_0_6": 190,
            "age_7_12": 175,
            "age_13_17": 145,
            "age_18_24": 230,
            "age_25_34": 360,
            "age_35_44": 370,
            "age_45_54": 340,
            "age_55_64": 300,
            "age_65_74": 240,
            "age_75_84": 165,
            "age_85_plus": 85,
        },
    ])
    paavo_demo.to_csv(paavo_input_path, index=False)

if not uk_households_input_path.exists():
    rows = []
    h_idx = 1
    random.seed(123)
    templates = [
        [34, 36, 8, 5], [72], [41, 39, 14], [29], [53, 52], [33, 31, 3],
        [24, 22, 0], [67, 65], [45, 43, 17, 13], [80], [58, 56, 26],
        [37, 35, 9], [49], [61, 59], [27, 25], [32, 30, 1],
    ]
    for _ in range(1500):
        hh = random.choice(templates)
        for age in hh:
            rows.append({"household_id": f"UK{h_idx:06d}", "age": age})
        h_idx += 1
    pd.DataFrame(rows).to_csv(uk_households_input_path, index=False)

print("Demo inputs ready.")
print("PAAVO rows:", len(pd.read_csv(paavo_input_path)))
print("UK household rows:", len(pd.read_csv(uk_households_input_path)))


## Load PAAVO Targets + UK Household Templates

In [ ]:
target_counts, postcode_metadata, paavo_harmonised_df = load_paavo_postcode_targets(
    paavo_input_path,
    postcode_col="postcode",
)

uk_table = pd.read_csv(uk_households_input_path)
uk_households = []
for _, grp in uk_table.groupby("household_id"):
    ages = [int(x) for x in grp["age"].tolist()]
    if ages:
        uk_households.append(ages)

print("Loaded postcodes:", len(target_counts))
print("Loaded household templates:", len(uk_households))
display(paavo_harmonised_df.head())


## Build Synthetic Population

In [ ]:
population = build_postcode_population(
    target_counts=target_counts,
    uk_households=uk_households,
    random_seed=random_seed,
    config=config,
    postcode_metadata=postcode_metadata,
)

print("Global summary:")
for k, v in population["global_summary"].items():
    if k == "config":
        continue
    print(f"  {k}: {v}")


## Postcode-Level Validation

In [ ]:
postcode_fit_df = population["postcode_fit_df"].copy()

display(postcode_fit_df[[
    "postcode", "postcode_name", "municipality", "latitude", "longitude",
    "target_population", "realised_population", "population_gap", "sse", "household_count",
    "sampling_iterations_used", "accepted_sampling_moves", "swap_attempts", "swap_accepts"
]])


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(postcode_fit_df["target_population"], postcode_fit_df["realised_population"], s=50)
axes[0].plot(
    [postcode_fit_df["target_population"].min(), postcode_fit_df["target_population"].max()],
    [postcode_fit_df["target_population"].min(), postcode_fit_df["target_population"].max()],
    linestyle="--",
    color="gray",
)
axes[0].set_title("Target vs Realised population")
axes[0].set_xlabel("Target")
axes[0].set_ylabel("Realised")
axes[0].grid(alpha=0.3)

households_df = population["households_df"]
axes[1].hist(households_df["household_size"], bins=range(1, 8), align="left", rwidth=0.8)
axes[1].set_title("Household size distribution")
axes[1].set_xlabel("Household size")
axes[1].set_ylabel("Count")
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()


## Individual-Level Output Preview

In [ ]:
individuals_df = population["individuals_df"]
households_df = population["households_df"]

display(individuals_df.head(10))
print("Individuals columns:", list(individuals_df.columns))
print("Households columns:", list(households_df.columns))


In [ ]:
# Example: age-bin comparison for one postcode
example_pc = postcode_fit_df.iloc[0]["postcode"]
row = postcode_fit_df[postcode_fit_df["postcode"] == example_pc].iloc[0]

labels = OPENABM_AGE_BIN_LABELS
target_vals = [row[f"target_{k}"] for k in labels]
real_vals = [row[f"realised_{k}"] for k in labels]

x = np.arange(len(labels))
width = 0.4

fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(x - width/2, target_vals, width=width, label="Target")
ax.bar(x + width/2, real_vals, width=width, label="Realised")
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_title(f"Age-bin fit for postcode {example_pc}")
ax.set_ylabel("Population")
ax.grid(axis="y", alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()
